# Objectives

- Understand the fundamental interactions with overcollateralized borrow-lending protocols
- Combining DeFi elements to perform a short on a token

# Prerequisites 

- Metamask 
- Ensure a minimum of 0.01 SepoliaETH plus extra for gas in your account - if not get it from [here](https://cloud.google.com/application/web3/faucet/ethereum/sepolia) 


# Interacting with Aave through Etherscan

Aave is one of the leading Borrow-lending platforms in DeFi. In this lab, we will borrow and lend some tokens on Aave V3 on the Sepolia testnet. 

We use Aave contracts on Sepolia testnet and call them directly through the Sepolia block explorer, Etherscan. This is exactly what the web app does under the hood when you click its buttons when working on the mainnet. The web app does not support testnet, that is why we have to call the contracts through explorer.

## Contract addresses (Ethereum Sepolia)

| Contract | Address |
|---|---|
| Aave V3 Pool | [0x6Ae43d3271ff6888e7Fc43Fd7321a503ff738951](https://sepolia.etherscan.io/address/0x6Ae43d3271ff6888e7Fc43Fd7321a503ff738951) |
| WrappedTokenGatewayV3 (ETH ↔ WETH helper) | [0x387d311e47e80b498169e6fb51d3193167d89F7D](https://sepolia.etherscan.io/address/0x387d311e47e80b498169e6fb51d3193167d89F7D) |
| USDC | [0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8](https://sepolia.etherscan.io/address/0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8) |
| aWETH (Aave wrapped ETH) | [0x5b071b590a59395fE4025A0Ccc1FcC931AAc1830](https://sepolia.etherscan.io/address/0x5b071b590a59395fE4025A0Ccc1FcC931AAc1830) |
| variableDebtUSDC | [0x36B5dE936eF1710E1d22EabE5231b28581a92ECc](https://sepolia.etherscan.io/address/0x36B5dE936eF1710E1d22EabE5231b28581a92ECc) |

## Using Etherscan to call a contract

- Open the contract address on Etherscan and go to the `Contract` tab
- The Pool and aWETH contracts are upgradeable proxies, so use the `Read as Proxy` / `Write as Proxy` tabs for them. For the Gateway and USDC, use `Read Contract` / `Write Contract`
- Before writing, click `Connect to Web3` and connect Metamask (make sure Metamask is on Sepolia)
- Token amounts are entered in the token's smallest unit: USDC has 6 decimals (1 USDC = `1000000`) and ETH/aWETH have 18 decimals (0.001 ETH = `1000000000000000`). The only exception is the `payableAmount` field, which is entered in ETH
- When a function asks for `onBehalfOf` or `to`, enter your own wallet address

## Collateralized borrow 

We aim to borrow USDC tokens using ETH tokens in our wallet as collateral. 
To borrow any funds from Aave, you need to supply collateral first. The contract accepts a broad set of pre-approved tokens as collateral. (Why do the tokens have to be pre-approved?). We will supply our ETH as collateral. Follow to step's below to collateralize your position:

- Before supplying, check your position on the [Pool → Read as Proxy](https://sepolia.etherscan.io/address/0x6Ae43d3271ff6888e7Fc43Fd7321a503ff738951#readProxyContract) → `getUserAccountData(user = your address)`. Everything should be zero since we don't have any collateral, so the amount available to borrow (`availableBorrowsBase`) is also zero
- To provide collateral, supply 0.01 ETH to the contract: on the **Gateway** (not the Pool) [Gateway → Write Contract](https://sepolia.etherscan.io/address/0x387d311e47e80b498169e6fb51d3193167d89F7D#writeContract) → `depositETH` with
  - `payableAmount` = `0.01`
  - the first field (`<input> (address)`, unnamed) = Pool address
  - `onBehalfOf` = your address
  - `referralCode` = `0`
- The Gateway wraps your ETH into WETH and supplies it to Aave for you. You will receive aWETH token (Aave wrapped ETH); add it to Metamask by importing the aWETH address above
- Call `getUserAccountData` again. The values are in USD with 8 decimals: `totalCollateralBase` should be `4000000000`, i.e. 40 USD. This does not correspond to the current price of ETH but rather a constant set in the testnet oracle. On the mainnet, the valuation of collateral is derived from a price-feed collateral
- Aave V3 allows us to borrow funds upto 80% of the ETH collateral value; check that `availableBorrowsBase` is `3200000000` (32 USD)
- Use these funds to borrow 2 USDCs at the variable interest rate: on the [Pool → Write as Proxy](https://sepolia.etherscan.io/address/0x6Ae43d3271ff6888e7Fc43Fd7321a503ff738951#writeProxyContract) → `borrow` with
  - `asset` = USDC address
  - `amount` = `2000000`
  - `interestRateMode` = `2` (variable)
  - `referralCode` = `0`
  - `onBehalfOf` = your address
- Not only do you get 2 USDC tokens but also AAVE variableDebtUSDC tokens (You can take a look later on how these tokens are used). Import both USDC and variableDebtUSDC into Metamask to see them, and call `getUserAccountData` again to see your debt and health factor

You can use the borrowed USDC as if they are your own tokens. You can exchange them on any AMM, transfer them to another account, or even supply them back to AAVE to earn interest. (Note that supply interest is always lower than borrow interest). Note that you aren't allowed to withdraw all of your ETH since you are required to repay your loan first

- Repay your borrowed position. Note that you will be required to submit more than 2 USDC since the borrowed USDC has accrued interest. For the sake of brevity of the lab, repay with 1 USDC. Repaying takes two transactions:
  - First allow the Pool to pull your USDC: on [USDC → Write Contract](https://sepolia.etherscan.io/address/0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8#writeContract) → `approve` with `spender` = Pool address and `amount` = `1000000`
  - Then on the [Pool → Write as Proxy](https://sepolia.etherscan.io/address/0x6Ae43d3271ff6888e7Fc43Fd7321a503ff738951#writeProxyContract) → `repay` with `asset` = USDC address, `amount` = `1000000`, `interestRateMode` = `2`, `onBehalfOf` = your address
  - (Why is the approval needed? The web app asks you to sign this same approval before a repay)
- You will now to able to withdraw almost all of your ETH. Since we still have a position open, we will only withdraw 0.001 ETH. The Gateway burns your aWETH to unwrap it back to ETH, so it needs an approval too:
  - On [aWETH → Write as Proxy](https://sepolia.etherscan.io/address/0x5b071b590a59395fE4025A0Ccc1FcC931AAc1830#writeProxyContract) → `approve` with `spender` = Gateway address and `amount` = `1000000000000000`
  - On the [Gateway → Write Contract](https://sepolia.etherscan.io/address/0x387d311e47e80b498169e6fb51d3193167d89F7D#writeContract) → `withdrawETH` with the first (unnamed) `address` field = Pool address, `amount` = `1000000000000000`, `to` = your address
- Observe that the transaction consumes aWETH and returns ETH to your wallet

## Performing a short using Aave

<span style="color:red">IMPORTANT - PERFORM THIS PART OF THE LAB IN CLASS ONLY </span>

Let's assume that you, as a trader, believe that USDC will lose its peg from USD 1 and go down to USD 0.9. Under this belief, going short on USDC will make the trader profit. 

Borrow-lending platforms can be used to perform a short on any token that we can borrow. In this part, we will be shorting 1 USDC. Please follow the steps below to perform a short.

Note your LUSD balance at the start of the experiment 
- Now we use 1 USDC from the borrowed amount
- Sell the 1 USDC for LUSD at the price of approximately 1 USDC/LUSD on Uniswap [link](https://app.uniswap.org/#/swap) (Use the addresses of USDC (0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8) and LUSD (0xa2039612e374ad2365361DF0FF34399A4f0A1e58))
- Use Sepolia and the Uniswap V3 0.05% [USDC/LUSD pool](https://sepolia.etherscan.io/address/0xb7e4986E4E43C75b012eE0bf804b3dbfed756E97).

The TA will now simulate the price dip of USDC by posting a large swap on Uniswap.

- Buy back the borrowed USDC plus accrued interest by paying LUSD. Since the price of USDC has dipped in reference to LUSD, the difference between LUSD consumed in this trade and the LUSD received in the previous trade is your profit! (Check your exact remaining debt with `balanceOf(your address)` on the variableDebtUSDC contract, and buy slightly more USDC than that since interest keeps accruing)
- Note the new LUSD balance - the increase in balance is roughly the profit from the short
- Repay all of the USDC debt on Aave (including interest) to close the short. Passing the maximum `uint256` value as the amount tells Aave to repay the entire debt:
  - On [USDC → Write Contract](https://sepolia.etherscan.io/address/0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8#writeContract) → `approve` with `spender` = Pool address and `amount` = `115792089237316195423570985008687907853269984665640564039457584007913129639935`
  - On the [Pool → Write as Proxy](https://sepolia.etherscan.io/address/0x6Ae43d3271ff6888e7Fc43Fd7321a503ff738951#writeProxyContract) → `repay` with `asset` = USDC address, `amount` = `115792089237316195423570985008687907853269984665640564039457584007913129639935`, `interestRateMode` = `2`, `onBehalfOf` = your address
  - Check that your variableDebtUSDC balance is now 0